# Challenge 5 — Boltz-2 co-folding validation

Predicts each candidate binder against the three targets the organizers measure:

| target | chains | goal |
|---|---|---|
| **G12D** | HLA-A\*11:01 + β2m + `VVVGADGVGK` | (c) strong binding |
| **WT**   | HLA-A\*11:01 + β2m + `VVVGAGGVGK` | (a) **no** binding |
| **apo**  | HLA-A\*11:01 + β2m               | (b) **no** binding |

Target sequences are verbatim from the challenge page (HLA 281 aa, β2m 99 aa).

Tamarind bills one job per predicted complex, so 60 predictions would cost 60 jobs.
Colab's T4 is free.

**Restart-safe.** Results append after every prediction and finished rows are skipped, so
a disconnect costs one prediction. Every candidate runs against G12D first, then WT, then
apo, so a partial run still leaves the column the submission most needs.

**Runtime** ≈ 3–6 min per prediction on a T4; 20 candidates × 3 targets ≈ 4–6 h.

## 1 · GPU and install

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q boltz==2.0.3 --no-deps
!pip install -q hydra-core pytorch-lightning rdkit modelcif mashumaro einops numba scipy biopython click pyyaml requests tqdm fairscale dm-tree einx gemmi chembl-structure-pipeline types-requests wandb
import os
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'  # newer torch blocks Boltz's checkpoint otherwise
import boltz, subprocess
print('boltz', boltz.__version__ if hasattr(boltz,'__version__') else
      subprocess.run(['pip','show','boltz'],capture_output=True,text=True).stdout.split('Version: ')[1].split('\n')[0])

## 2 · Working directory

Mount Drive — it is what makes a restart cheap.

In [ ]:
USE_DRIVE = True

import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = '/content/drive/MyDrive/ch5_boltz'
else:
    WORK = '/content/ch5_boltz'

for d in ('', '/yaml', '/out', '/msa'):
    os.makedirs(WORK + d, exist_ok=True)
RESULTS = f'{WORK}/results.csv'
print('working dir:', WORK)

## 3 · Targets (verbatim from the challenge page)

In [ ]:
HLA = ("GSHSMRYFYTSVSRPGRGEPRFIAVGYVDDTQFVRFDSDAASQRMEPRAPWIEQEGPEYWDQETRNVKAQSQTDRV"
       "DLGTLRGYYNQSEDGSHTIQIMYGCDVGPDGRFLRGYRQDAYDGKDYIALNEDLRSWTAADMAAQITKRKWEAAHA"
       "AEQQRAYLEGRCVEWLRRYLENGKETLQRTDPPKTHMTHHPISDHEATLRCWALGFYPAEITLTWQRDGEDQTQDT"
       "ELVETRPAGDGTFQKWAAVVVPSGEEQRYTCHVQHEGLPKPLTLRWELSSQPT")
B2M = "IQRTPKIQVYSRHPAENGKSNFLNCYVSGFHPSDIEVDLLKNGERIEKVEHSDLSFSKDWSFYLLYYTEFTPTEKDEYACRVNHVTLSQPKIVKWDRDM"
PEP_G12D = "VVVGADGVGK"
PEP_WT   = "VVVGAGGVGK"
assert len(HLA) == 281 and len(B2M) == 99

# Chain order here fixes the asym_id order Boltz uses in pair_chains_iptm.
TARGETS = {
    "G12D": [("A", HLA), ("B", B2M), ("C", PEP_G12D)],
    "WT":   [("A", HLA), ("B", B2M), ("C", PEP_WT)],
    "apo":  [("A", HLA), ("B", B2M)],
}
BINDER_CHAIN = "Z"
print({k: sum(len(s) for _, s in v) for k, v in TARGETS.items()})

## 4 · Candidate binders

Upload `ch5_mpnn_variants.csv` and, optionally, `ch5_candidates.csv` so the original RFD4
designs are predicted under identical settings as a baseline.

In [ ]:
import csv, os

N_MPNN, N_RFD4 = 20, 8

if not os.path.exists(f'{WORK}/ch5_mpnn_variants.csv'):
    from google.colab import files
    print('Upload ch5_mpnn_variants.csv (and ch5_candidates.csv):')
    for name, data in files.upload().items():
        open(f'{WORK}/{name}', 'wb').write(data)

load = lambda p: list(csv.DictReader(open(p)))
binders = []
p = f'{WORK}/ch5_mpnn_variants.csv'
if os.path.exists(p):
    binders += [{'name': r['name'], 'seq': r['sequence'], 'source': 'mpnn'}
                for r in load(p)[:N_MPNN]]
p = f'{WORK}/ch5_candidates.csv'
if os.path.exists(p):
    binders += [{'name': r['design_id'].replace('rfd4_design_', 'rfd4_d'),
                 'seq': r['sequence'], 'source': 'rfd4'}
                for r in load(p) if r.get('status') == 'candidate'][:N_RFD4]

bad = [b['name'] for b in binders if not 10 <= len(b['seq']) <= 250
       or set(b['seq']) - set('ACDEFGHIKLMNPQRSTVWY')]
assert not bad, f'competition length/alphabet rule violated: {bad}'
seen = set()
binders = [b for b in binders if not (b['seq'] in seen or seen.add(b['seq']))]
print(f'{len(binders)} binders -> {len(binders)*3} predictions')

## 5 · Target MSAs, fetched once

Boltz writes server MSAs into **each job's own** output folder, so `--use_msa_server` would
re-query for every prediction — 60 queries for 2 unique sequences, and a good way to get
rate-limited. One probe run harvests them; every later run points at the saved files.

Boltz writes these as **`.csv`** (`key,sequence`), not `.a3m`. Each file is matched to its
chain by comparing the query row to the sequence, so naming conventions cannot mislead us.

The peptide and the binder deliberately get no MSA: the peptide is 10 residues, and
single-sequence is the standard choice for a de novo binder.

In [ ]:
import os, glob, shutil, subprocess, csv as _csv

def harvest_msas():
    want = {'hla': HLA, 'b2m': B2M}
    have = {k: f'{WORK}/msa/{k}.csv' for k in want
            if os.path.exists(f'{WORK}/msa/{k}.csv')}
    if len(have) == len(want):
        print('MSAs cached:', have); return have

    probe = f'{WORK}/yaml/_msaprobe.yaml'
    with open(probe, 'w') as fh:
        fh.write('version: 1\nsequences:\n')
        for cid, seq in TARGETS['G12D']:
            fh.write(f'  - protein:\n      id: {cid}\n      sequence: {seq}\n')
    subprocess.run(['boltz', 'predict', probe, '--out_dir', f'{WORK}/out',
                    '--use_msa_server', '--diffusion_samples', '1',
                    '--recycling_steps', '1', '--no_trifast', '--override'], check=False)

    found = {}
    for f in glob.glob(f'{WORK}/out/boltz_results__msaprobe/msa/*.csv'):
        with open(f) as fh:
            rows = list(_csv.DictReader(fh))
        if not rows:
            continue
        query = rows[0]['sequence'].replace('-', '').upper()
        for tag, seq in want.items():
            if query == seq:
                dst = f'{WORK}/msa/{tag}.csv'
                shutil.copy(f, dst); found[tag] = dst
    print('harvested:', found)
    missing = set(want) - set(found)
    if missing:
        raise RuntimeError(f'MSA not harvested for {missing}. Do NOT continue: the target '
                           f'would silently run in single-sequence mode. Re-run this cell.')
    return found

MSA_FILES = harvest_msas()
MSA_FOR = {'A': MSA_FILES['hla'], 'B': MSA_FILES['b2m']}   # C and Z get none

## 6 · Predict, extract metrics, check clashes

In [ ]:
import json, glob, time, subprocess, warnings, re
import numpy as np
warnings.filterwarnings('ignore')
from Bio.PDB import MMCIFParser
from scipy.spatial import distance

KD = {'A':1.8,'R':-4.5,'N':-3.5,'D':-3.5,'C':2.5,'Q':-3.5,'E':-3.5,'G':-0.4,'H':-3.2,
      'I':4.5,'L':3.8,'K':-3.9,'M':1.9,'F':2.8,'P':-1.6,'S':-0.8,'T':-0.7,'W':-0.9,
      'Y':-1.3,'V':4.2}
BB = ('N', 'CA', 'C', 'O')

def write_yaml(path, chains, binder_seq):
    out = ['version: 1', 'sequences:']
    for cid, seq in chains:
        out += ['  - protein:', f'      id: {cid}', f'      sequence: {seq}']
        out.append(f'      msa: {MSA_FOR[cid]}' if cid in MSA_FOR else '      msa: empty')
    out += ['  - protein:', f'      id: {BINDER_CHAIN}', f'      sequence: {binder_seq}',
            '      msa: empty']
    open(path, 'w').write('\n'.join(out) + '\n')

def min_backbone_gap(cif, binder_chain=BINDER_CHAIN):
    # Smallest backbone-atom distance between binder and target.
    # Under 2.2 A the chains interpenetrate and it is not a complex.
    m = MMCIFParser(QUIET=True).get_structure('x', cif)[0]
    ids = [c.id for c in m]
    if binder_chain not in ids or len(ids) < 2:
        return float('nan')
    g = lambda cs: np.array([a.coord for c in cs for r in m[c] for a in r
                             if a.get_id() in BB])
    return float(distance.cdist(g([c for c in ids if c != binder_chain]),
                                g([binder_chain])).min())

def run_ipsae(pae_npz, cif):
    # Best-effort: ipsae writes .txt next to the inputs and prints a table.
    try:
        r = subprocess.run(['ipsae', pae_npz, cif, '10', '10'],
                           capture_output=True, text=True, timeout=600)
        vals = [float(x) for x in re.findall(r'\b0\.\d+\b', r.stdout)]
        return max(vals) if vals else float('nan')
    except Exception as e:
        print('  ipsae skipped:', e); return float('nan')

def predict(name, target, chains, binder_seq, seed=None):
    tag = f'{name}__{target}'
    y = f'{WORK}/yaml/{tag}.yaml'
    write_yaml(y, chains, binder_seq)
    t0 = time.time()
    subprocess.run(['boltz', 'predict', y, '--out_dir', f'{WORK}/out',
                    '--diffusion_samples', '1', '--recycling_steps', '3',
                    '--output_format', 'mmcif', '--write_full_pae', '--no_trifast', '--override']
                   + (['--seed', str(seed)] if seed is not None else []),
                   check=False)
    dt = round(time.time() - t0)

    d = f'{WORK}/out/boltz_results_{tag}/predictions/{tag}'
    conf = glob.glob(f'{d}/confidence_{tag}_model_0.json')
    cif = glob.glob(f'{d}/{tag}_model_0.cif')
    if not conf or not cif:
        return {'name': name, 'target': target, 'status': 'FAILED', 'seconds': dt}
    c = json.load(open(conf[0]))

    # pair_chains_iptm is keyed by asym_id INTEGERS (as strings in JSON), in the chain
    # order of the YAML - not by chain letter. Binder is the last chain.
    bi = str(len(chains))
    pair = c.get('pair_chains_iptm', {}) or {}
    def pc(i, j):
        try:    return float(pair[str(i)][str(j)])
        except Exception: return float('nan')

    pae = f'{d}/pae_{tag}_model_0.npz'
    row = {'name': name, 'target': target, 'status': 'ok', 'seconds': dt,
           'confidence': c.get('confidence_score'), 'iptm': c.get('iptm'),
           'ptm': c.get('ptm'), 'complex_plddt': c.get('complex_plddt'),
           'complex_ipde': c.get('complex_ipde'),
           'iptm_binder_HLA': pc(bi, 0), 'iptm_binder_b2m': pc(bi, 1),
           'iptm_binder_peptide': pc(bi, 2) if target != 'apo' else float('nan'),
           'ipsae': run_ipsae(pae, cif[0]) if os.path.exists(pae) else float('nan'),
           'mean_KD_hydropathy': round(float(np.mean([KD[a] for a in binder_seq])), 3),
           'binder_len': len(binder_seq), 'cif': cif[0]}
    g = min_backbone_gap(cif[0])
    row['min_backbone_gap_A'] = round(g, 2) if g == g else float('nan')
    row['clash_verdict'] = 'INTERPENETRATING' if g == g and g < 2.2 else 'ok'
    return row

print('ready')

## 7 · Main loop

In [ ]:
import csv, os

FIELDS = ['name', 'source', 'target', 'status', 'seconds', 'confidence', 'iptm', 'ptm',
          'complex_plddt', 'complex_ipde', 'iptm_binder_HLA', 'iptm_binder_b2m',
          'iptm_binder_peptide', 'ipsae', 'mean_KD_hydropathy', 'binder_len',
          'min_backbone_gap_A', 'clash_verdict', 'cif']

done = set()
if os.path.exists(RESULTS):
    done = {(r['name'], r['target']) for r in csv.DictReader(open(RESULTS))
            if r.get('status') == 'ok'}
    print(f'{len(done)} already done')
else:
    csv.DictWriter(open(RESULTS, 'w', newline=''), FIELDS).writeheader()

jobs = [(b, t) for t in ('G12D', 'WT', 'apo') for b in binders]
todo = [(b, t) for b, t in jobs if (b['name'], t) not in done]
print(f'{len(todo)} remaining of {len(jobs)}')

for i, (b, t) in enumerate(todo, 1):
    print(f'\n=== [{i}/{len(todo)}] {b["name"]} vs {t} ===', flush=True)
    row = predict(b['name'], t, TARGETS[t], b['seq'])
    row['source'] = b['source']
    with open(RESULTS, 'a', newline='') as fh:
        csv.DictWriter(fh, FIELDS, extrasaction='ignore').writerow(row)
    print({k: row.get(k) for k in ('iptm', 'iptm_binder_peptide', 'ipsae',
                                   'complex_plddt', 'min_backbone_gap_A', 'seconds')})
print('\nALL DONE')

## 7b · Seed replicates (only for the designs that discriminated)

Single-seed ipTM gaps can be noise: two sibling designs differed by 0.37 on the WT score.
This re-runs chosen designs with new seeds on all three targets. Rows are named
`<design>_s<seed>`; the loop is resumable. Edit `REPLICATE` / `SEEDS` as needed.

In [ ]:
REPLICATE = ['mpnn_d010_13', 'mpnn_d031_01']
SEEDS = [101, 202]

by_name = {b['name']: b for b in binders}
rep_jobs = [(by_name[n], t, sd) for sd in SEEDS for t in ('G12D', 'WT', 'apo')
            for n in REPLICATE if n in by_name]
done = {(r['name'], r['target']) for r in csv.DictReader(open(RESULTS))
        if r.get('status') == 'ok'}
todo = [(b, t, sd) for b, t, sd in rep_jobs if (f"{b['name']}_s{sd}", t) not in done]
print(f'{len(todo)} replicate predictions remaining of {len(rep_jobs)}')

for i, (b, t, sd) in enumerate(todo, 1):
    nm = f"{b['name']}_s{sd}"
    print(f'\n=== [{i}/{len(todo)}] {nm} vs {t} ===', flush=True)
    row = predict(nm, t, TARGETS[t], b['seq'], seed=sd)
    row['source'] = 'replicate'
    with open(RESULTS, 'a', newline='') as fh:
        csv.DictWriter(fh, FIELDS, extrasaction='ignore').writerow(row)
    print({k: row.get(k) for k in ('iptm', 'iptm_binder_peptide', 'complex_plddt',
                                   'min_backbone_gap_A', 'seconds')})
print('\nREPLICATES DONE')

## 7c · d010 family expansion

`mpnn_d010_13` is the only design whose G12D/WT gap survived three seeds. Its sibling
`mpnn_d010_14` shows a strong apo gap but a weak peptide gap on one seed. This cell asks
whether the d010 scaffold reliably yields discriminators or `_13` was a lucky draw:

* `mpnn_d010_15`, `mpnn_d010_16` — never tested, all three targets, default seed
* `mpnn_d010_14` — seeds 101 and 202, all three targets

12 predictions, ~1.1 h. Resumable.

In [ ]:
NEW_DESIGNS = ['mpnn_d010_15', 'mpnn_d010_16']
REP2        = ['mpnn_d010_14']
REP2_SEEDS  = [101, 202]

by_name = {b['name']: b for b in binders}
jobs2 = [(by_name[n], t, None) for t in ('G12D', 'WT', 'apo')
         for n in NEW_DESIGNS if n in by_name]
jobs2 += [(by_name[n], t, sd) for sd in REP2_SEEDS for t in ('G12D', 'WT', 'apo')
          for n in REP2 if n in by_name]

missing = [n for n in NEW_DESIGNS + REP2 if n not in by_name]
if missing:
    print('NOT FOUND in binders (check N_MPNN / the loader cell):', missing)

done = {(r['name'], r['target']) for r in csv.DictReader(open(RESULTS))
        if r.get('status') == 'ok'}
nm_of = lambda b, sd: b['name'] if sd is None else f"{b['name']}_s{sd}"
todo = [(b, t, sd) for b, t, sd in jobs2 if (nm_of(b, sd), t) not in done]
print(f'{len(todo)} remaining of {len(jobs2)}')

for i, (b, t, sd) in enumerate(todo, 1):
    nm = nm_of(b, sd)
    print(f'\n=== [{i}/{len(todo)}] {nm} vs {t} ===', flush=True)
    row = predict(nm, t, TARGETS[t], b['seq'], seed=sd)
    row['source'] = 'd010_family'
    with open(RESULTS, 'a', newline='') as fh:
        csv.DictWriter(fh, FIELDS, extrasaction='ignore').writerow(row)
    print({k: row.get(k) for k in ('iptm', 'iptm_binder_peptide', 'iptm_binder_HLA',
                                   'complex_plddt', 'min_backbone_gap_A', 'seconds')})
print('\nd010 FAMILY DONE')

## 7d · ipSAE (CPU only — no GPU quota)

ipSAE is not a pip package; it is a single script, which is why the earlier in-loop call
failed. This runs it over every prediction already on disk and writes `ipsae_scores.csv`.

ipSAE rescales the PAE-derived interface score by the size of the *interface* rather than
the whole complex, so it does not reward a large binder for incidental contact the way
raw ipTM can. Chains: **A** = HLA, **B** = β2m, **C** = peptide, **Z** = binder.

Safe to run on a CPU runtime. Takes a few minutes for ~60 predictions.

In [ ]:
import os, glob, csv, subprocess

if not os.path.exists('/content/IPSAE'):
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/DunbrackLab/IPSAE.git', '/content/IPSAE'],
                   check=True)

def parse_ipsae(txt, c1, c2):
    # Returns (ipSAE, pDockQ2) for the c1->c2 asym row, else (nan, nan).
    try:
        for line in open(txt):
            p = line.split()
            if len(p) > 11 and p[0] == c1 and p[1] == c2 and p[4] == 'asym':
                return float(p[5]), float(p[11])
    except Exception:
        pass
    return float('nan'), float('nan')

rows = []
dirs = sorted(glob.glob(f'{WORK}/out/boltz_results_*/predictions/*/'))
print(f'{len(dirs)} prediction folders')

for d in dirs:
    tag = os.path.basename(d.rstrip('/'))
    cif = glob.glob(f'{d}/{tag}_model_0.cif')
    pae = glob.glob(f'{d}/pae_{tag}_model_0.npz')
    if not cif or not pae:
        print('  skip (missing cif/pae):', tag); continue
    txt = cif[0].replace('.cif', '_10_10.txt')
    if not os.path.exists(txt):
        r = subprocess.run(['python', '/content/IPSAE/ipsae.py', pae[0], cif[0], '10', '10'],
                           capture_output=True, text=True, timeout=900)
        if not os.path.exists(txt):
            print('  FAILED:', tag, r.stderr.strip()[-200:]); continue
    name, target = tag.rsplit('__', 1)
    # ipSAE is asymmetric (Chn1 = aligned, Chn2 = scored), and for a 10-residue
    # peptide vs a ~80-120 residue binder the two directions differ a lot. Keep both.
    zc, zc_q = parse_ipsae(txt, 'Z', 'C')     # binder aligned, peptide scored
    cz, cz_q = parse_ipsae(txt, 'C', 'Z')     # peptide aligned, binder scored
    za, za_q = parse_ipsae(txt, 'Z', 'A')     # binder aligned, HLA scored
    az, _    = parse_ipsae(txt, 'A', 'Z')
    rows.append({'name': name, 'target': target,
                 'ipsae_Z_to_peptide': zc, 'ipsae_peptide_to_Z': cz,
                 'ipsae_peptide_max': max([v for v in (zc, cz) if v == v], default=float('nan')),
                 'pdockq2_peptide': max([v for v in (zc_q, cz_q) if v == v], default=float('nan')),
                 'ipsae_Z_to_HLA': za, 'ipsae_HLA_to_Z': az,
                 'ipsae_HLA_max': max([v for v in (za, az) if v == v], default=float('nan')),
                 'pdockq2_HLA': za_q})
    print(f'{tag:42s} pep {zc:6.3f}/{cz:6.3f}   HLA {za:6.3f}/{az:6.3f}')

out = f'{WORK}/ipsae_scores.csv'
with open(out, 'w', newline='') as fh:
    w = csv.DictWriter(fh, ['name', 'target', 'ipsae_Z_to_peptide', 'ipsae_peptide_to_Z',
                            'ipsae_peptide_max', 'pdockq2_peptide', 'ipsae_Z_to_HLA',
                            'ipsae_HLA_to_Z', 'ipsae_HLA_max', 'pdockq2_HLA'])
    w.writeheader(); w.writerows(rows)
print(f'\nwrote {len(rows)} rows -> {out}')

## 8 · Summary

In [ ]:
import csv
import numpy as np

rows = [r for r in csv.DictReader(open(RESULTS)) if r['status'] == 'ok']
by = {}
for r in rows:
    by.setdefault(r['name'], {})[r['target']] = r
f = lambda v: float(v) if v not in (None, '', 'nan') else float('nan')

out = []
for n, d in by.items():
    g, w, a = (f(d.get(t, {}).get('iptm', 'nan')) for t in ('G12D', 'WT', 'apo'))
    out.append((g - w, g - a, n, d, g, w, a))
out.sort(key=lambda x: (-(x[0] if x[0] == x[0] else -9), -x[4]))

print(f'{"name":16s} {"src":5s} {"G12D":>6s} {"WT":>6s} {"apo":>6s} {"dWT":>6s} '
      f'{"dapo":>6s} {"ipSAE":>6s} {"pLDDT":>6s} {"gap":>5s} {"KD":>6s}')
for dw, da, n, d, g, w, a in out:
    x = d.get('G12D', {})
    print(f'{n:16s} {x.get("source",""):5s} {g:6.3f} {w:6.3f} {a:6.3f} {dw:6.3f} '
          f'{da:6.3f} {f(x.get("ipsae","nan")):6.3f} {f(x.get("complex_plddt","nan")):6.1f} '
          f'{f(x.get("min_backbone_gap_A","nan")):5.2f} {f(x.get("mean_KD_hydropathy","nan")):6.2f}')

bad = [r['name'] for r in rows if r.get('clash_verdict') == 'INTERPENETRATING']
if bad:
    print('\nINTERPENETRATING, drop these regardless of score:', sorted(set(bad)))

print('\ndWT and dapo are ipTM(G12D) - ipTM(other); positive = prefers the mutant.')
print('These are predictor confidences, not affinities, and the predictor was never')
print('trained to resolve a single-residue difference. Read a small gap as no gap.')

In [ ]:
from google.colab import files
files.download(RESULTS)